# Approach 2 - train the v3 intent parser (GPU)

Self-contained: send this one notebook to a GPU machine (Kaggle T4 x2, or the H100), get back one zip.
Needs internet for `pip`, the SimuHome repo (data only, nothing of SimuHome is run) and the Qwen weights.

**Why retrain when the v2 parser already exists?** The v2 labels have no way to say "8 minutes after the
previous action" (42 of 50 QT4-1 feasible requests) or "no later than" (all QT4-3 requests), so a v2
parser writes those as "8 minutes from now". That never changes a contradiction verdict, but it would make
the new start-time check (gate C1) refuse correct schedules. v3 labels fix both; everything else about the
parser (model, LoRA, data split) is unchanged, so v2 and v3 results are comparable.

**Output** (`/kaggle/working/parser_v3_outputs.zip`, or `./parser_v3_outputs.zip` elsewhere):
`qwen3_ir_parser_adapter_v3/`, `parser_eval_v3.json`, `parser_errors_v3.json`, `train_log_v3.json`,
`data/qt4_ir_v3_*.jsonl`, and **`parse_cache_v3.json`** (the parser's output for all 300 QT4 requests).
Attach the parse cache to the E1 notebook and E1 needs no GPU at all.


## 1. Setup


In [ ]:
# Separate installs: if pip cannot resolve one package set it installs NOTHING from that line (and -q hides
# the error), so z3 must not depend on the training libraries resolving.
!pip install -q z3-solver
!pip install -q -U transformers peft accelerate trl datasets
# Kaggle ships torchao 0.10; recent peft refuses any torchao older than 0.16 when it adds the LoRA layers
# (ImportError: incompatible version of torchao). We never use torchao, so remove it.
!pip uninstall -y -q torchao
import importlib.util
_missing = [m for m in ('z3', 'transformers', 'peft', 'trl', 'datasets', 'accelerate') if importlib.util.find_spec(m) is None]
assert not _missing, f'not installed: {_missing}. Read the pip output above (rerun the two lines without -q).'
import os, sys
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')   # one GPU is plenty; avoids DataParallel on 2 x T4
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
os.chdir(WORK)
if not os.path.isdir('SimuHome'):
    !git clone --depth 1 https://github.com/holi-lab/SimuHome.git
BENCH = 'SimuHome/data/benchmark'
import glob
_n = len(glob.glob(f'{BENCH}/qt4-*.json'))
assert _n == 300, f'expected 300 QT4 episodes in {BENCH}, found {_n}: the clone is missing or incomplete (delete SimuHome/ and rerun)'
sys.path.insert(0, '.')


In [ ]:
import os; os.makedirs('irlib', exist_ok=True); open('irlib/__init__.py', 'a').close()


In [ ]:
%%writefile irlib/ir_schema.py
"""Typed intermediate representation (IR v3) for SimuHome QT4 scheduling requests.

An IR describes one `schedule_workflow`-shaped user request as a set of goals.
Each goal is a set of target device assertions gated by one or more *time
readings* (ways the natural-language query says when the action happens).
A feasible request has one consistent reading per goal; an infeasible one states
two readings that resolve to different times for the same action (SimuHome's
"Contradiction Blindness" failure mode).

Changes from v2 (approach 1):
  - Anchor kind "goal": a reading measured from an EARLIER goal of the same
    request ("8 minutes after the previous action"). v2 had no such anchor and
    labelled these readings as "now"-anchored, which is harmless for a pure
    contradiction check but wrong for checking the agent's start_time.
  - Relation "by": a deadline (the state must hold no later than t). SimuHome
    marks these goals with tolerance_ticks == 0 and relation "before" (e.g. QT4-3
    "start the dryer now ... pause it when the washer finishes": the dryer only
    has to be running before the washer ends). v2 treated them as a point.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Literal, Optional

Relation = Literal["after", "before", "at", "by"]
AnchorKind = Literal["now", "device_event", "goal"]

IR_VERSION = 3


@dataclass
class Anchor:
    """What a time offset is measured from."""

    kind: AnchorKind = "now"
    room_id: Optional[str] = None
    device_id: Optional[str] = None
    device_type: Optional[str] = None
    goal_ref: Optional[int] = None  # kind == "goal": index of the earlier goal

    def to_dict(self) -> dict[str, Any]:
        return {
            "kind": self.kind,
            "room_id": self.room_id,
            "device_id": self.device_id,
            "device_type": self.device_type,
            "goal_ref": self.goal_ref,
        }


@dataclass
class TimeReading:
    """One way the query expresses *when* an action should occur.

    All times are minutes since the episode's `base_time` (the moment the user
    spoke). A second reading on the same goal that disagrees with the first is
    what makes a request infeasible.
    """

    relation: Relation
    anchor: Anchor
    offset_minutes: Optional[float] = None  # relative reading
    absolute_time: Optional[str] = None  # "HH:MM[:SS]", clock-time reading
    # device_event anchors only: when the anchor device finishes (minutes since
    # base_time). Never in the training target; filled from live simulator state.
    resolved_at_minutes: Optional[float] = None
    tolerance_minutes: float = 0.0

    def to_dict(self) -> dict[str, Any]:
        return {
            "relation": self.relation,
            "anchor": self.anchor.to_dict(),
            "offset_minutes": self.offset_minutes,
            "absolute_time": self.absolute_time,
            "resolved_at_minutes": self.resolved_at_minutes,
            "tolerance_minutes": self.tolerance_minutes,
        }


@dataclass
class Assertion:
    attribute: str
    value: Any
    value_type: Optional[str] = None
    description: Optional[str] = None

    def to_dict(self) -> dict[str, Any]:
        return {
            "attribute": self.attribute,
            "value": self.value,
            "value_type": self.value_type,
            "description": self.description,
        }


@dataclass
class Target:
    room_id: str
    device_id: str
    device_type: Optional[str]
    asserts: list[Assertion] = field(default_factory=list)

    def to_dict(self) -> dict[str, Any]:
        return {
            "room_id": self.room_id,
            "device_id": self.device_id,
            "device_type": self.device_type,
            "asserts": [a.to_dict() for a in self.asserts],
        }


@dataclass
class Goal:
    """One scheduled effect: a set of device assertions gated by time reading(s)."""

    goal_id: int
    readings: list[TimeReading]
    targets: list[Target]
    # Set when the conflict is a duration/resource issue rather than a stated-time
    # mismatch (e.g. "finish the washer exactly when the dishwasher finishes" but the
    # washer cycle is longer). Not checkable without a device duration model.
    requires_duration_model: bool = False

    def to_dict(self) -> dict[str, Any]:
        return {
            "goal_id": self.goal_id,
            "readings": [r.to_dict() for r in self.readings],
            "targets": [t.to_dict() for t in self.targets],
            "requires_duration_model": self.requires_duration_model,
        }


@dataclass
class IntentIR:
    query: str
    query_type: str
    case: Literal["feasible", "infeasible", "unknown"]
    base_time: str  # "YYYY-MM-DD HH:MM:SS"
    goals: list[Goal] = field(default_factory=list)
    conflict_type: Optional[str] = None  # ground-truth label, training-time only

    def to_dict(self) -> dict[str, Any]:
        return {
            "query": self.query,
            "query_type": self.query_type,
            "case": self.case,
            "base_time": self.base_time,
            "goals": [g.to_dict() for g in self.goals],
            "conflict_type": self.conflict_type,
        }


def ir_from_dict(d: dict) -> IntentIR:
    """Inverse of IntentIR.to_dict() (used for the debug ground-truth IRs)."""
    goals = []
    for g in d["goals"]:
        readings = [
            TimeReading(anchor=Anchor(**r["anchor"]), **{k: v for k, v in r.items() if k != "anchor"})
            for r in g["readings"]
        ]
        targets = [
            Target(
                room_id=t["room_id"],
                device_id=t["device_id"],
                device_type=t.get("device_type"),
                asserts=[Assertion(**a) for a in t.get("asserts", [])],
            )
            for t in g["targets"]
        ]
        goals.append(
            Goal(
                goal_id=g["goal_id"],
                readings=readings,
                targets=targets,
                requires_duration_model=g.get("requires_duration_model", False),
            )
        )
    return IntentIR(
        query=d["query"],
        query_type=d["query_type"],
        case=d["case"],
        base_time=d["base_time"],
        goals=goals,
        conflict_type=d.get("conflict_type"),
    )


# ---------------------------------------------------------------------------
# Compact wire format (what the SLM is trained to emit)
# ---------------------------------------------------------------------------
# Keeps only what the query text determines: per goal, the time readings and the
# targets. Z3 decides feasibility; the model never says it. v3 adds `goal_ref`
# for goal anchors and the "by" relation; everything else is unchanged from v2,
# so a v2 parser's output still loads (it just never produces goal anchors).

DEFAULT_TOLERANCE_MINUTES = 0.5


def compact_ir_dict(ir_dict: dict, *, keep_resolved: bool = False) -> dict:
    """Full IR dict -> compact dict. Clock times are cut to HH:MM (the query
    states minutes, never the simulator's seconds)."""
    goals = []
    for g in ir_dict["goals"]:
        readings = []
        for r in g["readings"]:
            anchor = {"kind": r["anchor"]["kind"]}
            if r["anchor"].get("device_id"):
                anchor["device_id"] = r["anchor"]["device_id"]
            if r["anchor"].get("room_id"):
                anchor["room_id"] = r["anchor"]["room_id"]
            if r["anchor"].get("goal_ref") is not None:
                anchor["goal_ref"] = r["anchor"]["goal_ref"]
            out = {"relation": r["relation"], "anchor": anchor}
            if r.get("offset_minutes") is not None:
                out["offset_minutes"] = r["offset_minutes"]
            if r.get("absolute_time") is not None:
                out["absolute_time"] = ":".join(r["absolute_time"].split(":")[:2])
            if keep_resolved and r.get("resolved_at_minutes") is not None:
                out["resolved_at_minutes"] = r["resolved_at_minutes"]
            readings.append(out)
        targets = [
            {
                "device_id": t["device_id"],
                "asserts": [{"attribute": a["attribute"], "value": a["value"]} for a in t.get("asserts", [])],
            }
            for t in g["targets"]
        ]
        goals.append({"readings": readings, "targets": targets})
    return {"goals": goals}


def ir_from_compact(compact: dict, query: str, base_time: str) -> IntentIR:
    """Compact dict (+ the live query and the episode base_time) -> IntentIR."""
    goals = []
    for i, g in enumerate(compact["goals"]):
        readings = []
        for r in g["readings"]:
            a = r.get("anchor") or {}
            readings.append(
                TimeReading(
                    relation=r.get("relation", "after"),
                    anchor=Anchor(
                        kind=a.get("kind", "now"),
                        room_id=a.get("room_id"),
                        device_id=a.get("device_id"),
                        goal_ref=a.get("goal_ref"),
                    ),
                    offset_minutes=r.get("offset_minutes"),
                    absolute_time=r.get("absolute_time"),
                    resolved_at_minutes=r.get("resolved_at_minutes"),
                    tolerance_minutes=r.get("tolerance_minutes", DEFAULT_TOLERANCE_MINUTES),
                )
            )
        targets = [
            Target(
                room_id=t.get("room_id", ""),
                device_id=t["device_id"],
                device_type=None,
                asserts=[Assertion(attribute=a["attribute"], value=a["value"]) for a in t.get("asserts", [])],
            )
            for t in g.get("targets", [])
        ]
        goals.append(Goal(goal_id=i, readings=readings, targets=targets))
    return IntentIR(query=query, query_type="qt4", case="unknown", base_time=base_time, goals=goals)


def uses_goal_anchors(compact: dict) -> bool:
    """True if a compact IR contains any v3 goal anchor (i.e. came from a v3 parser
    or from v3 labels). Used to decide whether start_time checks are trustworthy."""
    return any(
        (r.get("anchor") or {}).get("kind") == "goal"
        for g in compact.get("goals", [])
        for r in g.get("readings", [])
    )


In [ ]:
%%writefile irlib/constraint_encoder.py
"""Component 2: IR -> Z3 constraints over real-valued time; SAT/UNSAT and time windows.

Time is one Real variable per goal: minutes since the episode's `base_time` (the
moment the user spoke, "now" = 0). Each TimeReading adds one constraint.

Changes from approach 1 (which solved every goal separately):
  - All goals are encoded in ONE solver, so readings can refer to each other:
    goal anchors ("8 min after the previous action") and a device anchor shared
    by several goals ("10 and 20 min after the dishwasher finishes" must agree on
    when the dishwasher finishes).
  - "by" readings are deadlines (t <= target + tol), not points.
  - When the request is consistent, `goal_windows` returns the interval [lo, hi]
    of start times each goal allows. The gate checks the agent's
    `schedule_workflow(start_time=...)` against these windows. This is the part a
    one-line Python comparison cannot do once goals chain off each other.

CPU-only; no GPU, simulator or SLM needed to run or test this module.
"""

from __future__ import annotations

import functools
import threading
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Optional

import z3

from .ir_schema import Goal, IntentIR, TimeReading

# Z3's default context is not thread-safe: two episodes checking at the same time
# (the parallel runner) crash it with "unreachable" / "invalid argument". Every
# public entry point holds this lock; a check takes milliseconds, so it costs nothing.
_Z3_LOCK = threading.RLock()


def _z3_serialized(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        with _Z3_LOCK:
            return fn(*args, **kwargs)
    return wrapper


def _parse_hms_to_minutes_of_day(hms: str) -> float:
    parts = [int(x) for x in hms.split(":")]
    h, m = parts[0], parts[1]
    s = parts[2] if len(parts) > 2 else 0
    return h * 60.0 + m + s / 60.0


def _is_minute_granular(hms: str) -> bool:
    return len(hms.split(":")) < 3


def _base_time_minutes_of_day(base_time: str) -> float:
    return _parse_hms_to_minutes_of_day(base_time.split(" ", 1)[1])


def minutes_to_clock(base_time: str, minutes: float) -> str:
    """Minutes since base_time -> "HH:MM:SS" wall-clock string."""
    base = datetime.strptime(base_time, "%Y-%m-%d %H:%M:%S")
    return (base + timedelta(minutes=float(minutes))).strftime("%H:%M:%S")


def clock_to_minutes(base_time: str, timestamp: str) -> float:
    """"YYYY-MM-DD HH:MM:SS" -> minutes since base_time."""
    base = datetime.strptime(base_time, "%Y-%m-%d %H:%M:%S")
    return (datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S") - base).total_seconds() / 60.0


def reading_to_absolute_minutes(reading: TimeReading, base_time: str) -> Optional[float]:
    """Minutes since base_time for a self-contained clock-time reading, else None."""
    if reading.absolute_time is not None and reading.anchor.kind == "now":
        # Clock times are same-day in SimuHome QT4 episodes; no midnight wrap handling.
        return _parse_hms_to_minutes_of_day(reading.absolute_time) - _base_time_minutes_of_day(base_time)
    return None


@dataclass
class GoalCheckResult:
    goal_id: int
    sat: bool
    reason: Optional[str] = None  # populated on UNSAT (or "skipped: ...")
    unsat_core: Optional[list[str]] = None
    resolved_minutes: Optional[float] = None  # one consistent time, on SAT


@dataclass
class GoalWindow:
    """Start times (minutes since base_time) a goal allows. None = unbounded."""

    goal_id: int
    lo: Optional[float]
    hi: Optional[float]
    deadline: bool = False  # only an upper bound was stated ("by")

    def contains(self, minutes: float, slack: float = 0.0) -> bool:
        if self.lo is not None and minutes < self.lo - slack:
            return False
        if self.hi is not None and minutes > self.hi + slack:
            return False
        return True


@dataclass
class FeasibilityResult:
    feasible: bool
    goal_results: list[GoalCheckResult]
    base_time: Optional[str] = None
    core_descriptions: list[str] = field(default_factory=list)

    def explanation(self) -> str:
        bad = [g for g in self.goal_results if not g.sat]
        if not bad:
            return "All scheduling constraints are consistent."
        lines = [f"Goal {g.goal_id}: {g.reason}" for g in bad]
        return "Infeasible schedule - " + "; ".join(lines)


def _describe_reading(reading: TimeReading, base_time: Optional[str], value: Optional[float]) -> str:
    when = f" (= {minutes_to_clock(base_time, value)})" if (base_time and value is not None) else ""
    if reading.absolute_time is not None:
        return f"stated clock time {reading.absolute_time}"
    if reading.offset_minutes is None:
        return "unspecified time reading"
    if reading.anchor.kind == "now":
        anchor_desc = "now"
    elif reading.anchor.kind == "goal":
        anchor_desc = f"goal {reading.anchor.goal_ref}"
    else:
        anchor_desc = f"{reading.anchor.device_id} finishing"
    rel = "before" if reading.relation == "by" else reading.relation
    prefix = "no later than " if reading.relation == "by" else ""
    return f"{prefix}{reading.offset_minutes:g} min {rel} {anchor_desc}{when}"


class _Encoding:
    """Builds the joint constraint system once; reused for the check and the windows."""

    def __init__(self, ir: IntentIR):
        self.ir = ir
        self.t = {g.goal_id: z3.Real(f"t_goal_{g.goal_id}") for g in ir.goals}
        self.anchors: dict[str, z3.ArithRef] = {}
        self.pins: list[tuple[str, z3.BoolRef]] = []  # known anchor times (not "readings")
        self.readings: list[tuple[str, z3.BoolRef, int, TimeReading, Optional[float]]] = []
        self.skipped: dict[int, str] = {}
        for goal in ir.goals:
            self._encode_goal(goal)

    def _anchor_var(self, device_id: str, resolved: Optional[float], label: str) -> z3.ArithRef:
        if device_id not in self.anchors:
            self.anchors[device_id] = z3.Real(f"anchor_{device_id}")
        var = self.anchors[device_id]
        if resolved is not None:
            self.pins.append((f"{label}_anchor_known", var == z3.RealVal(resolved)))
        return var

    def _encode_goal(self, goal: Goal) -> None:
        if goal.requires_duration_model:
            self.skipped[goal.goal_id] = "skipped: requires a device duration model (not checked)"
            return
        t = self.t[goal.goal_id]
        for i, reading in enumerate(goal.readings):
            label = f"g{goal.goal_id}_r{i}"
            extra = 1.0 if (reading.absolute_time and _is_minute_granular(reading.absolute_time)) else 0.0
            # An "HH:MM" clock time is only known to the minute, so widen by one minute.
            tol = z3.RealVal(reading.tolerance_minutes + extra)
            sign = -1 if reading.relation in ("before", "by") else 1
            nominal: Optional[float] = None  # the reading's own time, when known, for explanations

            if reading.anchor.kind == "device_event":
                anchor_var = self._anchor_var(reading.anchor.device_id or "?", reading.resolved_at_minutes, label)
                if reading.offset_minutes is None:
                    continue
                offset = 0.0 if reading.relation == "at" else sign * reading.offset_minutes
                target = anchor_var + z3.RealVal(offset)
                if reading.resolved_at_minutes is not None:
                    nominal = reading.resolved_at_minutes + offset
            elif reading.anchor.kind == "goal":
                ref = reading.anchor.goal_ref
                if ref is None or ref not in self.t or ref == goal.goal_id or reading.offset_minutes is None:
                    continue  # malformed reference (e.g. a parser slip): no constraint rather than a wrong one
                target = self.t[ref] + z3.RealVal(sign * reading.offset_minutes)
            else:
                resolved = reading_to_absolute_minutes(reading, self.ir.base_time)
                if resolved is not None:
                    target = z3.RealVal(resolved)
                    nominal = resolved
                elif reading.offset_minutes is not None:
                    target = z3.RealVal(sign * reading.offset_minutes)
                    nominal = sign * reading.offset_minutes
                else:
                    continue

            if reading.relation == "by":
                constraint = t <= target + tol
            else:
                constraint = z3.And(t >= target - tol, t <= target + tol)
            self.readings.append((label, constraint, goal.goal_id, reading, nominal))

    def solver(self, track: bool) -> z3.Solver:
        s = z3.Solver()
        for label, c in self.pins:
            if track:
                s.assert_and_track(c, z3.Bool(label))
            else:
                s.add(c)
        for label, c, *_ in self.readings:
            if track:
                s.assert_and_track(c, z3.Bool(label))
            else:
                s.add(c)
        return s


@_z3_serialized
def check_feasibility(ir: IntentIR) -> FeasibilityResult:
    enc = _Encoding(ir)
    solver = enc.solver(track=True)
    if solver.check() == z3.sat:
        model = solver.model()
        results = []
        for g in ir.goals:
            if g.goal_id in enc.skipped:
                results.append(GoalCheckResult(goal_id=g.goal_id, sat=True, reason=enc.skipped[g.goal_id]))
                continue
            val = model.eval(enc.t[g.goal_id], model_completion=True)
            results.append(GoalCheckResult(goal_id=g.goal_id, sat=True, resolved_minutes=float(val.as_fraction())))
        return FeasibilityResult(feasible=True, goal_results=results, base_time=ir.base_time)

    core = {str(c) for c in solver.unsat_core()}
    by_label = {label: (gid, r, nominal) for label, _, gid, r, nominal in enc.readings}
    bad_goals: dict[int, list[str]] = {}
    descriptions = []
    # "*_anchor_known" pins only fix a symbolic anchor to a runtime-known value;
    # they are not a statement in the request, so they are not named.
    for label in sorted(core):
        if label in by_label:
            gid, r, nominal = by_label[label]
            d = _describe_reading(r, ir.base_time, nominal)
            bad_goals.setdefault(gid, []).append(d)
            descriptions.append(f"goal {gid}: {d}")
    results = []
    for g in ir.goals:
        if g.goal_id in bad_goals:
            results.append(
                GoalCheckResult(
                    goal_id=g.goal_id,
                    sat=False,
                    reason=" vs. ".join(bad_goals[g.goal_id]),
                    unsat_core=sorted(core),
                )
            )
        else:
            results.append(GoalCheckResult(goal_id=g.goal_id, sat=True, reason=enc.skipped.get(g.goal_id)))
    if not bad_goals:  # core made only of pins (should not happen); still report UNSAT honestly
        results[0] = GoalCheckResult(goal_id=results[0].goal_id, sat=False, reason="unsatisfiable", unsat_core=sorted(core))
    return FeasibilityResult(feasible=False, goal_results=results, base_time=ir.base_time, core_descriptions=descriptions)


def _bound(enc: _Encoding, var: z3.ArithRef, maximize: bool) -> Optional[float]:
    opt = z3.Optimize()
    for _, c in enc.pins:
        opt.add(c)
    for _, c, *_ in enc.readings:
        opt.add(c)
    h = opt.maximize(var) if maximize else opt.minimize(var)
    if opt.check() != z3.sat:
        return None
    value = opt.upper(h) if maximize else opt.lower(h)
    # Z3 hands back an integer or rational numeral when the bound is finite, and a
    # non-numeral term (oo, epsilon) when the goal is unbounded in that direction.
    if z3.is_int_value(value):
        return float(value.as_long())
    if z3.is_rational_value(value):
        return float(value.as_fraction())
    return None


@_z3_serialized
def goal_windows(ir: IntentIR) -> dict[int, GoalWindow]:
    """For a consistent IR: the [lo, hi] start-time interval of every goal.

    Goals whose readings are all unresolved (e.g. the anchor device is idle, so its
    finish time is unknown) come back unbounded, and the gate does not check them.
    """
    enc = _Encoding(ir)
    if enc.solver(track=False).check() != z3.sat:
        return {}
    windows = {}
    for g in ir.goals:
        if g.goal_id in enc.skipped:
            windows[g.goal_id] = GoalWindow(goal_id=g.goal_id, lo=None, hi=None)
            continue
        t = enc.t[g.goal_id]
        lo, hi = _bound(enc, t, maximize=False), _bound(enc, t, maximize=True)
        deadline = all(r.relation == "by" for r in g.readings) and bool(g.readings)
        windows[g.goal_id] = GoalWindow(goal_id=g.goal_id, lo=lo, hi=hi, deadline=deadline)
    return windows


In [ ]:
%%writefile irlib/extract_ir_from_episode.py
"""Component: deterministic (query, IR) pair extraction from SimuHome episodes.

No LLM calls, no manual labeling. SimuHome benchmark/generated episodes
already carry, in `eval.goals` (ground-truth required behavior) and
`temporal_conflict` (ground-truth contradiction, infeasible episodes only),
every field needed to derive the IR mechanically. See handoff Sec 5.3.

Two IRs are produced per episode:
  - `ir_for_training`: what the SLM must learn to produce from the query
    text alone. Device-anchored readings never carry a numeric resolved
    time here, because the query text does not state one — the live
    simulator state does, and only the runtime agent can fetch it.
  - `debug_ground_truth`: the same IR with anchor times filled in from the
    simulator-computed fields, for exercising the Z3 encoder end-to-end
    without needing a live simulator.

IR v3 (approach 2) differs from v2 in two labelling rules, both read off
`eval.goals[].when`:
  - a goal with no device anchor but `reference_minutes > 0` is chained to the
    earlier goal whose `at_minutes` equals that reference ("8 minutes after the
    previous action") -> Anchor(kind="goal", goal_ref=j);
  - `relation == "before"` with `tolerance_ticks == 0` is a deadline (the state
    must hold by then, e.g. QT4-3 "start the dryer now") -> relation "by".
"""

from __future__ import annotations

import glob
import json
import os
from typing import Any, Optional

from .ir_schema import Anchor, Assertion, Goal, IntentIR, Target, TimeReading, compact_ir_dict

# Conflict types where the contradiction is a stated-time mismatch (checkable
# from two TimeReadings alone). See handoff Sec 6: duration/resource
# feasibility conflicts (e.g. "finish washer exactly when dishwasher ends"
# but the washer cycle itself is longer) need a device duration model the
# Z3 encoder does not have yet — flagged, not silently mis-scored.
_DURATION_MODEL_CONFLICT_TYPES = {"completion_vs_pause"}


def _tolerance_minutes(when: dict[str, Any], tick_interval: float) -> float:
    return float(when.get("tolerance_ticks", 0)) * tick_interval / 60.0


def _chained_goal_ref(
    goal_id: int, when: dict[str, Any], raw_goals: list[dict[str, Any]], *, any_direction: bool = False
) -> Optional[int]:
    """Index of the goal this one is measured from, or None.

    SimuHome writes "8 minutes after the previous action" as relation "after",
    offset 8, with `reference_minutes` = the previous goal's `at_minutes` and no
    device anchor; the latest earlier goal with that time is the reference.
    Deadline goals (QT4-3, `any_direction=True`) reference the OTHER goal's time
    ("the dryer must be running 1 min before it is paused"), which can be a later goal.
    """
    ref = when.get("reference_minutes")
    if not ref:
        return None
    for j in range(goal_id - 1, -1, -1):
        if raw_goals[j]["when"].get("at_minutes") == ref:
            return j
    if any_direction:
        for j in range(goal_id + 1, len(raw_goals)):
            if raw_goals[j]["when"].get("at_minutes") == ref:
                return j
    return None


def _device_anchor(raw_anchor: dict[str, Any]) -> Anchor:
    return Anchor(
        kind="device_event",
        room_id=raw_anchor.get("room_id"),
        device_id=raw_anchor.get("device_id"),
        device_type=raw_anchor.get("device_type"),
    )


def _goal_from_eval_goal(
    goal_id: int,
    raw_goal: dict[str, Any],
    tick_interval: float,
    raw_goals: Optional[list[dict[str, Any]]] = None,
    for_training: bool = True,
) -> Goal:
    when = raw_goal["when"]
    raw_anchor = raw_goal.get("anchor") or {}

    relation = when.get("relation", "after")
    deadline = relation == "before" and int(when.get("tolerance_ticks", 0)) == 0
    if deadline:
        relation = "by"  # deadline: the state must hold no later than anchor - offset

    goal_ref = _chained_goal_ref(goal_id, when, raw_goals or [], any_direction=deadline)
    if deadline and goal_ref is not None:
        # Measured from the other goal, even when eval.goals labels a device anchor:
        # its reference_minutes is that goal's time, not the device's finish.
        anchor = Anchor(kind="goal", goal_ref=goal_ref)
    elif raw_anchor.get("device_id"):
        anchor = _device_anchor(raw_anchor)
    elif goal_ref is not None:
        anchor = Anchor(kind="goal", goal_ref=goal_ref)
    else:
        anchor = Anchor(kind="now")

    reading = TimeReading(
        relation=relation,
        anchor=anchor,
        offset_minutes=float(when.get("offset_minutes", 0.0)),
        tolerance_minutes=_tolerance_minutes(when, tick_interval),
    )
    if anchor.kind == "device_event" and not for_training and when.get("reference_tick") is not None:
        # Debug IR only: when the anchor device finishes, as the runtime reads it from
        # the simulator (CountdownTime). The training IR never contains it. Ticks, not
        # reference_minutes, which SimuHome rounds to whole minutes.
        reading.resolved_at_minutes = float(when["reference_tick"]) * tick_interval / 60.0

    targets = [
        Target(
            room_id=t["room_id"],
            device_id=t["device_id"],
            device_type=t.get("device_type"),
            asserts=[
                Assertion(
                    attribute=a["attribute"],
                    value=a["value"],
                    value_type=a.get("value_type"),
                    description=a.get("description"),
                )
                for a in t.get("asserts", [])
            ],
        )
        for t in raw_goal.get("targets", [])
    ]

    return Goal(goal_id=goal_id, readings=[reading], targets=targets)


def _inject_conflicting_reading(
    goal: Goal, temporal_conflict: dict[str, Any], for_training: bool, raw_anchor: Optional[dict[str, Any]] = None
) -> Goal:
    """Add the second, contradicting TimeReading to the primary goal.

    Mutates and returns `goal`. If the conflict is duration/resource-based
    (no comparable second time reading exists), flags the goal instead.
    """
    conflict_type = temporal_conflict.get("type")

    if conflict_type in _DURATION_MODEL_CONFLICT_TYPES or "conflict_at" not in temporal_conflict:
        goal.requires_duration_model = True
        return goal

    anchor_end_at = temporal_conflict.get("anchor_end_at")
    expected_at = temporal_conflict.get("expected_at")
    conflict_time = temporal_conflict.get("conflict_time")
    relation = temporal_conflict.get("relation", "after")

    if anchor_end_at is not None:
        # Dependency-anchored primary reading (e.g. "20 min after dishwasher finishes").
        base_offset = temporal_conflict.get(
            "offset_minutes", temporal_conflict.get("delay_minutes", 0.0)
        )
        offset = float(base_offset)
        # The conflict is stated against the anchor DEVICE finishing, so use the
        # device from eval.goals even if v3 relabelled the goal as goal-anchored.
        anchor = _device_anchor(raw_anchor) if (raw_anchor or {}).get("device_id") else goal.readings[0].anchor
        primary_reading = TimeReading(
            relation=relation,
            anchor=anchor,
            offset_minutes=offset,
            tolerance_minutes=goal.readings[0].tolerance_minutes,
            # resolved_at_minutes here is the ANCHOR EVENT's own resolved
            # time (e.g. when the dishwasher finishes), not the target time
            # — the Z3 encoder derives target = anchor +/- offset itself.
            resolved_at_minutes=None if for_training else float(anchor_end_at),
        )
        goal.readings[0] = primary_reading
    elif expected_at is not None:
        # Now-anchored primary reading (e.g. "13 minutes from now").
        goal.readings[0].offset_minutes = float(expected_at)
        goal.readings[0].anchor = Anchor(kind="now")

    if conflict_time is not None:
        goal.readings.append(
            TimeReading(
                relation="at",
                anchor=Anchor(kind="now"),
                absolute_time=conflict_time.split(" ")[-1] if " " in conflict_time else conflict_time,
                tolerance_minutes=goal.readings[0].tolerance_minutes,
            )
        )

    return goal


def extract_episode(episode: dict[str, Any], *, for_training: bool = True) -> IntentIR:
    """Build an IntentIR from one raw SimuHome benchmark/generated episode dict."""
    meta = episode["meta"]
    tick_interval = float(episode["initial_home_config"]["tick_interval"])
    base_time = episode["initial_home_config"]["base_time"]
    case = meta.get("case", "unknown")

    raw_goals = episode["eval"]["goals"]
    goals = [
        _goal_from_eval_goal(i, rg, tick_interval, raw_goals, for_training) for i, rg in enumerate(raw_goals)
    ]

    conflict_type = None
    if case == "infeasible" and "temporal_conflict" in episode:
        temporal_conflict = episode["temporal_conflict"]
        conflict_type = temporal_conflict.get("type")
        first = goals[0].readings[0] if goals and goals[0].readings else None
        if (
            first is not None
            and first.relation == "by"
            and first.anchor.kind == "goal"
            and conflict_type not in _DURATION_MODEL_CONFLICT_TYPES
            and temporal_conflict.get("conflict_time")
        ):
            # QT4-3: every goal keeps exactly the reading its FEASIBLE twin has; the stated
            # clock time is added to the goal it is about. v3.0 instead replaced goal 0's
            # reading, so feasible and infeasible QT4-3 labels differed in more than the clock
            # time, and the parser copied the infeasible template (with an invented clock time)
            # onto feasible requests (held-out seeds 55, 75).
            #   absolute_time_mismatch: "start at 6:14 PM ... pause when X finishes" -> the start (goal 0)
            #   delay_time_mismatch:    "start 14 min after X finishes, at 11:40"  -> the delayed start
            #   impossible_early_end:   "finish together before 2:35 PM"           -> the finish
            if conflict_type == "absolute_time_mismatch":
                target = goals[0]
            else:
                target = next((g for g in goals if g.readings and g.readings[0].anchor.kind == "device_event"), goals[0])
            clock = temporal_conflict["conflict_time"]
            target.readings.append(
                TimeReading(
                    relation="at",
                    anchor=Anchor(kind="now"),
                    absolute_time=clock.split(" ")[-1],
                    tolerance_minutes=target.readings[0].tolerance_minutes,
                )
            )
        elif goals:
            goals[0] = _inject_conflicting_reading(
                goals[0], temporal_conflict, for_training, raw_goals[0].get("anchor")
            )

    return IntentIR(
        query=episode["query"],
        query_type=meta.get("query_type", "unknown"),
        case=case,
        base_time=base_time,
        goals=goals,
        conflict_type=conflict_type,
    )


def extract_pair(episode: dict[str, Any]) -> dict[str, Any]:
    """Return {"query", "ir_for_training", "debug_ground_truth", "meta"} for one episode."""
    training_ir = extract_episode(episode, for_training=True)
    debug_ir = extract_episode(episode, for_training=False)
    return {
        "query": episode["query"],
        "ir_for_training": compact_ir_dict(training_ir.to_dict()),
        "debug_ground_truth": debug_ir.to_dict(),
        "meta": episode["meta"],
    }


def extract_dataset(benchmark_dir: str, pattern: str = "qt4-*_seed_*.json") -> list[dict[str, Any]]:
    """Extract (query, IR) pairs for every QT4 episode file in `benchmark_dir`."""
    # Benchmark files sit directly in the dir; generator output is <run>/episodes/*.json.
    paths = sorted(
        glob.glob(os.path.join(benchmark_dir, "qt4-*.json"))
        + glob.glob(os.path.join(benchmark_dir, "*", "episodes", "qt4-*.json"))
    )
    pairs = []
    errors = []
    for path in paths:
        try:
            with open(path, "r", encoding="utf-8") as f:
                episode = json.load(f)
            pair = extract_pair(episode)
            pair["source_file"] = os.path.basename(path)
            pairs.append(pair)
        except Exception as e:  # keep going; report at the end
            errors.append((path, str(e)))
    if errors:
        print(f"[extract_dataset] {len(errors)} file(s) failed to extract:")
        for path, err in errors[:20]:
            print(f"  {os.path.basename(path)}: {err}")
    return pairs


if __name__ == "__main__":
    import sys

    bench_dir = sys.argv[1] if len(sys.argv) > 1 else "SimuHome/data/benchmark"
    out_path = sys.argv[2] if len(sys.argv) > 2 else "data/qt4_ir_pairs.jsonl"
    pairs = extract_dataset(bench_dir)
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for pair in pairs:
            f.write(json.dumps(pair, ensure_ascii=False) + "\n")
    print(f"Extracted {len(pairs)} pairs -> {out_path}")


In [ ]:
%%writefile irlib/grounding.py
"""Grounding filter: drop clock times the parser wrote that the user never said.

The v3 parser sometimes adds an absolute clock reading that is not in the request
(e.g. "at 16:00" for "start the dishwasher now"). Z3 then finds a genuine clash
with the other reading and calls a POSSIBLE request impossible. A clock time is
something the user states explicitly, so it can be checked against the sentence
deterministically: if no time in the text matches it, the reading is removed
before Z3 sees it. Relative offsets are not filtered; a stated time plus/minus an offset in the same
IR also counts as stated (see ground_ir).

Times are recognised in digit form ("12:23 PM", "1:39 pm", "12 36 PM", "7 PM") and
word form ("six fifty six PM", "twelve thirty six PM", "seven o'clock").
"""

from __future__ import annotations

import copy
import re

_UNITS = {
    "zero": 0, "oh": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8,
    "nine": 9, "ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14, "fifteen": 15,
    "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19,
}
_TENS = {"twenty": 20, "thirty": 30, "forty": 40, "fifty": 50}


def _with_meridiem(h: int, m: int, mer: str | None) -> set[tuple[int, int]]:
    if not (0 <= h <= 23 and 0 <= m <= 59):
        return set()
    if mer:
        mer = mer.replace(".", "").lower()
        if mer == "pm" and h < 12:
            h += 12
        elif mer == "am" and h == 12:
            h = 0
        return {(h, m)}
    out = {(h, m)}
    if h < 12:
        out.add((h + 12, m))  # no AM/PM given: either reading
    return out


def _words_to_numbers(text: str) -> str:
    """'six fifty six pm' -> '6 56 pm'; 'twelve thirty six' -> '12 36'."""
    tokens = re.findall(r"[a-z]+|\d+|[^a-z\d\s]", text.lower())
    out, i = [], 0
    while i < len(tokens):
        t = tokens[i]
        if t in _TENS:
            val = _TENS[t]
            if i + 1 < len(tokens) and tokens[i + 1] in _UNITS and 0 < _UNITS[tokens[i + 1]] < 10:
                val += _UNITS[tokens[i + 1]]
                i += 1
            out.append(str(val))
        elif t in _UNITS:
            val = _UNITS[t]
            if t == "oh" and i + 1 < len(tokens) and tokens[i + 1] in _UNITS:  # "ten oh three" -> "10 03"
                out.append(f"{_UNITS[tokens[i + 1]]:02d}")
                i += 2
                continue
            out.append(str(val))
        else:
            out.append(t)
        i += 1
    return " ".join(out)


def times_in_text(text: str) -> set[tuple[int, int]]:
    """Every (hour 0-23, minute) the text could be stating as a clock time."""
    found: set[tuple[int, int]] = set()
    for variant in (text.lower(), _words_to_numbers(text)):
        for h, m, mer in re.findall(r"\b(\d{1,2})\s*[:.\s]\s*(\d{2})\s*(a\.?m\.?|p\.?m\.?)?", variant):
            found |= _with_meridiem(int(h), int(m), mer or None)
        for h, mer in re.findall(r"\b(\d{1,2})\s*(a\.?m\.?|p\.?m\.?)", variant):
            found |= _with_meridiem(int(h), 0, mer)
        for h in re.findall(r"\b(\d{1,2})\s*o\s*'?\s*clock", variant):
            found |= _with_meridiem(int(h), 0, None)
    return found


def _closest_device(name: str, known: list[str], cutoff: float) -> str | None:
    """Closest real device id, preferring the same room (the id prefix before the device type)."""
    import difflib

    same_room = [k for k in known if k.split("_")[0] == name.split("_")[0]]
    for pool in (same_room, known):
        hit = difflib.get_close_matches(name, pool, n=1, cutoff=cutoff)
        if hit:
            return hit[0]
    return None


def repair_device_ids(compact: dict, known_ids: list[str], cutoff: float = 0.85) -> tuple[dict, int, int]:
    """Copy of `compact` whose device ids that do not exist in the house are replaced by the closest real
    one (e.g. the parser's 'utility_laundry_dryer_1' -> 'utility_room_laundry_dryer_1'). Uses only the
    house's device list, never the answer key. Returns (ir, n_repaired, n_unresolved)."""
    known = list(known_ids)
    known_set = set(known)
    out = copy.deepcopy(compact)
    repaired = unresolved = 0

    def fix(obj: dict) -> None:
        nonlocal repaired, unresolved
        dev = obj.get("device_id")
        if not dev or dev in known_set:
            return
        hit = _closest_device(dev, known, cutoff)
        if hit:
            obj["device_id"] = hit
            repaired += 1
        else:
            unresolved += 1

    for g in out.get("goals", []):
        for r in g.get("readings", []):
            if (r.get("anchor") or {}).get("kind") == "device_event":
                fix(r["anchor"])
        for t in g.get("targets", []):
            fix(t)
    return out, repaired, unresolved


def ground_ir(compact: dict, query: str) -> tuple[dict, int]:
    """Copy of `compact` without clock readings the query does not state. Returns (ir, n_dropped).

    A clock time counts as stated if it appears in the text, or if it is a stated time
    plus/minus an offset the same IR uses ("finishes around 4:42 PM ... start 19 minutes
    after" -> 17:01): the benchmark's own labels express such derived times this way.
    """
    stated = times_in_text(query)
    offsets = {int(round(r["offset_minutes"])) for g in compact.get("goals", []) for r in g.get("readings", [])
               if isinstance(r.get("offset_minutes"), (int, float))}
    derived = set()
    for h, m in stated:
        for o in offsets:
            for t in (h * 60 + m + o, h * 60 + m - o):
                derived.add(((t // 60) % 24, t % 60))
    stated = stated | derived
    out = copy.deepcopy(compact)
    dropped = 0
    for g in out.get("goals", []):
        kept = []
        for r in g.get("readings", []):
            at = r.get("absolute_time")
            if at:
                try:
                    h, m = (int(x) for x in at.split(":")[:2])
                except ValueError:
                    dropped += 1
                    continue
                if (h, m) not in stated:
                    dropped += 1
                    continue
            kept.append(r)
        g["readings"] = kept
    return out, dropped


## 2. Check the v3 labels before training

With the ground-truth IR the joint Z3 encoder must still reach every benchmark verdict (286/286, the 14
duration-model conflicts excepted), and every benchmark goal time must lie inside the window it computes.
Same check as `tests/test_pipeline.py`.


In [ ]:
import glob, json
from irlib.extract_ir_from_episode import extract_episode, extract_dataset
from irlib.constraint_encoder import check_feasibility, goal_windows
ok = n = inside = goals = 0
for path in sorted(glob.glob(f'{BENCH}/qt4-*.json')):
    ep = json.load(open(path, encoding='utf-8'))
    if ep['meta'].get('conflict_type') == 'completion_vs_pause':
        continue
    ir = extract_episode(ep, for_training=False)
    n += 1; ok += check_feasibility(ir).feasible == (ep['meta']['case'] == 'feasible')
    if ep['meta']['case'] == 'feasible':
        w = goal_windows(ir); tick = float(ep['initial_home_config']['tick_interval'])
        for gi, g in enumerate(ep['eval']['goals']):
            goals += 1; inside += w[gi].contains(g['when']['at_tick'] * tick / 60.0, 1e-6)
print(f'verdicts {ok}/{n}, goal times inside their window {inside}/{goals}')
assert n == 286 and goals == 301 and ok == n and inside == goals, (n, ok, goals, inside)


## 3. Training data (same split as v2)

The 300 native QT4 episodes; every seed divisible by 5 is held out (54 episodes, the E1 test set); the rest
are shuffled with seed 0 and split 90/10 into train (221) and validation (25). Identical to approach 1.


In [ ]:
import random
os.makedirs('data', exist_ok=True)
native = extract_dataset(BENCH)
held_out = [p for p in native if int(p['meta']['seed']) % 5 == 0]
trainable = [p for p in native if int(p['meta']['seed']) % 5 != 0]
random.seed(0); random.shuffle(trainable)
k = int(len(trainable) * 0.9)
train_rows, val_rows, test_rows = trainable[:k], trainable[k:], held_out
assert not ({p['query'] for p in train_rows + val_rows} & {p['query'] for p in test_rows}), 'split leak'
for name, rows in [('train', train_rows), ('val', val_rows), ('test_held_out', test_rows)]:
    with open(f'data/qt4_ir_v3_{name}.jsonl', 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
print(len(train_rows), 'train', len(val_rows), 'val', len(test_rows), 'held-out')
print('example v3 label:', json.dumps(next(r for r in train_rows if 'goal_ref' in json.dumps(r['ir_for_training'])), ensure_ascii=False)[:600])


## 4. LoRA fine-tune

Same recipe as v2 (Qwen3-1.7B, LoRA r=16 on all attention and MLP projections, 3 epochs, lr 2e-4).
`MODELS` can list several base models; on an H100 add `Qwen/Qwen3-4B` and `Qwen/Qwen3-0.6B` for a
parser-size comparison in the paper (each takes minutes there). The first entry's adapter is the one
used for the parse cache. Checkpoints are saved, so a killed session resumes.


In [ ]:
# ============================ EDIT THIS BLOCK ONLY ============================
MODELS = ['Qwen/Qwen3-1.7B']   # T4: keep one model. H100: ['Qwen/Qwen3-1.7B', 'Qwen/Qwen3-4B', 'Qwen/Qwen3-0.6B']
CACHE_ALL = False  # False = only the 54 held-out requests E1 uses (~15 min on a T4). True = all 300, but the other
                   # 246 were TRAINING data for this parser, so their parses must not be used to score anything.
EPOCHS, LR, LORA_R = 5, 2e-4, 16   # v3.0 used 3 epochs; its validation loss was still falling (0.71, 0.49, 0.45)
# ==============================================================================
import torch, inspect
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
SYSTEM_PROMPT = (
    'You convert a smart-home scheduling request into a strict JSON '
    'intermediate representation (IR). Output ONLY the JSON object, '
    'no prose, no code fences.'
)
def fmt(row):
    return {'text': f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{row['query']}\n<|assistant|>\n" + json.dumps(row['ir_for_training'], ensure_ascii=False)}
train_ds = Dataset.from_list([fmt(r) for r in train_rows]); val_ds = Dataset.from_list([fmt(r) for r in val_rows])
# Precision by GPU generation. bf16 needs compute capability >= 8.0 (A100/H100); current torch/peft no longer
# emulate it on a T4 (7.5) and fail with 'gemm input type BFloat16 ... compute capability 8.0'. On a T4: fp32
# weights + fp16 autocast (fp16=True), the standard safe recipe; the LoRA weights stay fp32, so fp16 gradient
# scaling works. Inference on a T4 also uses fp32 (fp16 generation can overflow; correctness over speed).
CC = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
BF16 = CC[0] >= 8
MODEL_DTYPE = torch.bfloat16 if BF16 else torch.float32
print(f'GPU compute capability {CC}: weights {MODEL_DTYPE}, mixed precision', 'bf16' if BF16 else 'fp16')
big_gpu = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 40e9
def adapter_dir(model_name):
    tag = model_name.split('/')[-1].lower().replace('.', '_')
    return 'qwen3_ir_parser_adapter_v3' if model_name == MODELS[0] else f'ir_parser_adapter_v3_{tag}'

import hashlib, shutil
# Fingerprint of everything that defines a trained adapter: labels (hash of the training data), recipe, model.
# An adapter or checkpoint folder is reused ONLY if its fingerprint matches. (v3.1's first run silently reused
# the v3.0 adapter left in /kaggle/working by a previous session, because the old check was 'folder exists'.)
LABELS_HASH = hashlib.sha256(json.dumps([r['ir_for_training'] for r in train_rows + val_rows], sort_keys=True).encode()).hexdigest()[:12]
def run_tag(model_name):
    return f'{model_name}|labels={LABELS_HASH}|epochs={EPOCHS}|lr={LR}|r={LORA_R}|best_by_eval_loss'

def _tag_matches(folder, tag):
    f = os.path.join(folder, 'TRAINED_WITH.txt')
    return os.path.exists(f) and open(f).read().strip() == tag

def train(model_name):
    out = adapter_dir(model_name)
    tag, ckpt_dir = run_tag(model_name), f'checkpoints_{out}'
    if os.path.isdir(out) and _tag_matches(out, tag):
        print('SKIPPED training (an adapter with exactly these labels/settings exists):', out); return out
    for stale in (out, ckpt_dir):   # trained with other labels or settings: must not be reused or resumed
        if os.path.isdir(stale) and not _tag_matches(stale, tag):
            print('removing stale', stale); shutil.rmtree(stale)
    os.makedirs(ckpt_dir, exist_ok=True)
    open(os.path.join(ckpt_dir, 'TRAINED_WITH.txt'), 'w').write(tag)
    print('TRAINING', tag)
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=MODEL_DTYPE, device_map={'': 0})
    lora = LoraConfig(r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                      target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
    len_kw = 'max_length' if 'max_length' in inspect.signature(SFTConfig.__init__).parameters else 'max_seq_length'
    bs, acc = (16, 1) if big_gpu else (2, 8)   # effective batch 16 either way; fp32 weights on a T4 need the smaller step
    cfg = SFTConfig(output_dir=f'checkpoints_{out}', per_device_train_batch_size=bs, gradient_accumulation_steps=acc,
                    num_train_epochs=EPOCHS, learning_rate=LR, logging_steps=5, save_strategy='epoch', save_total_limit=2,
                    eval_strategy='epoch', bf16=BF16, fp16=not BF16, report_to=[], dataset_text_field='text',
                    loss_type='nll', load_best_model_at_end=True, metric_for_best_model='eval_loss',
                    greater_is_better=False, **{len_kw: 1024})   # keeps the epoch with the lowest validation loss
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, eval_dataset=val_ds, peft_config=lora)
    ckpts = sorted(glob.glob(f'checkpoints_{out}/checkpoint-*'))
    trainer.train(resume_from_checkpoint=ckpts[-1] if ckpts else None)
    json.dump(trainer.state.log_history, open(f'train_log_{out}.json', 'w'), indent=2)
    trainer.model.save_pretrained(out)
    open(os.path.join(out, 'TRAINED_WITH.txt'), 'w').write(tag)
    AutoTokenizer.from_pretrained(model_name).save_pretrained(out)
    del trainer, model; torch.cuda.empty_cache()
    return out

ADAPTERS = {m: train(m) for m in MODELS}
print(ADAPTERS)


## 5. Evaluate on the held-out episodes

Besides valid JSON, exact match and **verdict accuracy** (does Z3 on the predicted IR give the right
feasible/infeasible answer, device anchors filled with the simulator's times as at runtime), v3 reports:
- **goals wrongly refused** (the key safety number): on feasible held-out episodes each benchmark goal is
  matched to the predicted goals by device, as the gate does; if its correct time is outside every matching
  window, the gate's time check would refuse a correct schedule. Should be near 0.
- **goals unmatched**: the parser named a device wrongly, so the gate cannot check that goal (lets it through).
- **clock times dropped**: times the parser invented that the grounding filter removed (as the gate does).
- **target accuracy**: are the predicted target devices and attribute values exactly the benchmark's? This
  decides whether the dry run's target check (C2) should warn or reject.


In [ ]:
from irlib.ir_schema import ir_from_compact
def load_parser(model_name, adapter):
    tok = AutoTokenizer.from_pretrained(adapter)
    base = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=MODEL_DTYPE, device_map={'': 0})
    return tok, PeftModel.from_pretrained(base, adapter).eval()

def make_parse(tok, model):
    def parse(query, max_new_tokens=700):
        prompt = f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{query}\n<|assistant|>\n"
        ids = tok(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
        return json.loads(tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True))
    return parse

def fill_anchors(pred, gold):
    # Device finish times come from the simulator at runtime; here from the gold IR, matched by DEVICE id
    # (not by goal position: the parser may list the same goals in a different order).
    finish = {r['anchor']['device_id']: r.get('resolved_at_minutes') for g in gold['goals'] for r in g['readings']
              if r['anchor'].get('kind') == 'device_event' and r.get('resolved_at_minutes') is not None}
    for g in pred['goals']:
        for r in g['readings']:
            a = r.get('anchor') or {}
            if a.get('kind') == 'device_event':
                r['resolved_at_minutes'] = finish.get(a.get('device_id'))

from irlib.grounding import ground_ir
def eval_parser(parse, rows, bench=BENCH):
    # Scores the parser the way the gate uses it: clock-time grounding on, and each benchmark goal matched
    # to predicted goals by DEVICE (as the gate matches a schedule_workflow call), not by position.
    s = dict(n=0, valid=0, exact=0, verdict_ok=0, verdict_n=0, clock_dropped=0,
             goals=0, goals_ok=0, goals_unmatched=0, goals_refused=0, targets_exact=0, targets_n=0)
    failures = []
    for row in rows:
        s['n'] += 1
        try:
            pred = parse(row['query'])
            s['valid'] += 1
        except Exception:
            failures.append((row['query'], 'invalid JSON')); continue
        s['exact'] += pred == row['ir_for_training']
        gold_t = [t for g in row['ir_for_training']['goals'] for t in g['targets']]
        try:
            pred_t = [t for g in pred['goals'] for t in g['targets']]
        except Exception:
            pred_t = None
        s['targets_n'] += 1; s['targets_exact'] += pred_t == gold_t
        gold = row['debug_ground_truth']
        if gold['conflict_type'] == 'completion_vs_pause':
            continue
        try:
            pred, dropped = ground_ir(pred, row['query'])
            s['clock_dropped'] += dropped
            fill_anchors(pred, gold)
            ir = ir_from_compact(pred, row['query'], gold['base_time'])
            verdict = check_feasibility(ir).feasible
        except Exception as e:
            failures.append((row['query'], f'bad structure: {type(e).__name__}')); continue
        s['verdict_n'] += 1
        want = row['meta']['case'] == 'feasible'
        s['verdict_ok'] += verdict == want
        if verdict != want:
            failures.append((row['query'], 'wrong verdict: predicted ' + ('feasible' if verdict else 'infeasible')))
        if want and verdict:
            ep = json.load(open(bench + '/' + row['source_file'], encoding='utf-8'))
            tick = float(ep['initial_home_config']['tick_interval'])
            w = goal_windows(ir)
            for gi, g in enumerate(ep['eval']['goals']):
                s['goals'] += 1
                t, devs = g['when']['at_tick'] * tick / 60.0, {x['device_id'] for x in g['targets']}
                cands = [pg for pg in ir.goals if any(tt.device_id in devs for tt in pg.targets)]
                if not cands:
                    s['goals_unmatched'] += 1   # wrong device name: the gate cannot check it and lets it through
                elif any(w.get(pg.goal_id) is None or w[pg.goal_id].contains(t, 0.25) for pg in cands):
                    s['goals_ok'] += 1
                else:
                    s['goals_refused'] += 1
                    failures.append((row['query'], f'goal {gi}: a CORRECT schedule would be refused by the time check'))
    return s, failures

results, all_failures = {}, {}
for m in MODELS:
    tok, model = load_parser(m, ADAPTERS[m])
    parse = make_parse(tok, model)
    for split, rows in [('val', val_rows), ('test', test_rows)]:
        s, fails = eval_parser(parse, rows)
        results[f'{m}|{split}'] = s
        if split == 'test':
            all_failures[m] = [{'query': q, 'why': w} for q, w in fails]
        print(f"{m:22s} {split:4s} valid {s['valid']}/{s['n']} | exact {s['exact']}/{s['n']} | verdict {s['verdict_ok']}/{s['verdict_n']} | "
              f"goals: ok {s['goals_ok']}, WRONGLY REFUSED {s['goals_refused']}, unmatched {s['goals_unmatched']} of {s['goals']} | "
              f"clock times dropped {s['clock_dropped']} | targets exact {s['targets_exact']}/{s['targets_n']}")
    if m != MODELS[0]:
        del model; torch.cuda.empty_cache()
    else:
        main_tok, main_model = tok, model
json.dump(results, open('parser_eval_v3.json', 'w'), indent=2)
json.dump(all_failures, open('parser_errors_v3.json', 'w'), indent=2, ensure_ascii=False)


## 6. Parse cache for E1

The parser's output depends only on the request text, so parse every QT4 request once here. With this file
attached, the E1 notebook runs on CPU and never loads the parser. Each entry also records the parse time,
which the paper reports as the gate's latency overhead.


In [ ]:
import time
parse = make_parse(main_tok, main_model)
cache = {'_meta': {'ir_version': 3, 'labels': '3.1', 'run_tag': run_tag(MODELS[0]), 'adapter_sha256': hashlib.sha256(open(os.path.join(ADAPTERS[MODELS[0]], 'adapter_model.safetensors'), 'rb').read()).hexdigest()[:16], 'model': MODELS[0], 'adapter': ADAPTERS[MODELS[0]], 'created': time.strftime('%Y-%m-%d %H:%M')}}
for row in (native if CACHE_ALL else test_rows):
    t0 = time.time()
    try:
        cache[row['query']] = {'ir': parse(row['query']), 'seconds': round(time.time() - t0, 3)}
    except Exception as e:
        cache[row['query']] = {'error': repr(e)[:200], 'seconds': round(time.time() - t0, 3)}
json.dump(cache, open('parse_cache_v3.json', 'w'), ensure_ascii=False)
print(len(cache) - 1, 'requests parsed;', sum('error' in v for k, v in cache.items() if k != '_meta'), 'failures')


In [ ]:
import zipfile
OUT = os.path.join(WORK, 'parser_v3_outputs.zip')
keep = ['parse_cache_v3.json', 'parser_eval_v3.json', 'parser_errors_v3.json', 'data'] + glob.glob('train_log_*.json') + list(ADAPTERS.values())
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in keep:
        if os.path.isfile(p):
            z.write(p)
        elif os.path.isdir(p):
            for root, _, files in os.walk(p):
                for fn in files:
                    z.write(os.path.join(root, fn))
print(OUT, round(os.path.getsize(OUT) / 1e6, 1), 'MB')
